In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


In [3]:
df = pd.read_csv("/Users/krishkalsi/Desktop/JPMC_QR_Sim/Task 3 and 4_Loan_Data.csv")

df.head()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0


In [4]:
features = [
    "credit_lines_outstanding",
    "loan_amt_outstanding",
    "total_debt_outstanding",
    "income",
    "years_employed",
    "fico_score"
]

X = df[features]
y = df["default"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [5]:
pd_model = LogisticRegression(max_iter=1000)
pd_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [6]:
pd_preds = pd_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, pd_preds)
auc

0.9999851558267091

In [7]:
LGD = 0.9  # 90% loss given default

def expected_loss(
    credit_lines_outstanding,
    loan_amt_outstanding,
    total_debt_outstanding,
    income,
    years_employed,
    fico_score
):
    """
    Returns expected loss for a single borrower.
    """

    borrower_data = pd.DataFrame([{
        "credit_lines_outstanding": credit_lines_outstanding,
        "loan_amt_outstanding": loan_amt_outstanding,
        "total_debt_outstanding": total_debt_outstanding,
        "income": income,
        "years_employed": years_employed,
        "fico_score": fico_score
    }])

    pd_estimate = pd_model.predict_proba(borrower_data)[0, 1]

    EL = pd_estimate * LGD * loan_amt_outstanding
    return pd_estimate, EL


In [8]:
pd_est, el = expected_loss(
    credit_lines_outstanding=5,
    loan_amt_outstanding=5000,
    total_debt_outstanding=12000,
    income=30000,
    years_employed=2,
    fico_score=570
)

pd_est, el


(0.9999999999998805, 4499.9999999994625)

In [11]:
import numpy as np
import pandas as pd

df = df.sort_values("fico_score").reset_index(drop=True)

fico = df["fico_score"].values
default = df["default"].values
N = len(df)

In [12]:
def log_likelihood(k, n):
    if k == 0 or k == n:
        return 0.0
    p = k / n
    return k * np.log(p) + (n - k) * np.log(1 - p)


LL = np.zeros((N, N))

for i in range(N):
    k = 0
    for j in range(i, N):
        k += default[j]
        n = j - i + 1
        LL[i, j] = log_likelihood(k, n)

In [13]:
def optimal_buckets(K):
    dp = np.full((K + 1, N), -np.inf)
    prev = np.zeros((K + 1, N), dtype=int)

    dp[1] = LL[0]

    for k in range(2, K + 1):
        for j in range(k - 1, N):
            for i in range(k - 2, j):
                val = dp[k - 1, i] + LL[i + 1, j]
                if val > dp[k, j]:
                    dp[k, j] = val
                    prev[k, j] = i

    return dp, prev

In [14]:
def recover_boundaries(prev, K):
    boundaries = []
    j = N - 1
    for k in range(K, 1, -1):
        i = prev[k, j]
        boundaries.append(fico[i])
        j = i
    boundaries.reverse()
    return boundaries

In [15]:
K = 5  # example

dp, prev = optimal_buckets(K)
boundaries = recover_boundaries(prev, K)

boundaries

[521, 580, 640, 696]

In [16]:
def fico_to_rating(fico_score, boundaries):
    for i, b in enumerate(boundaries):
        if fico_score <= b:
            return i + 1
    return len(boundaries) + 1